In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# TEMP
# warnings.simplefilter("ignore", category=FutureWarning)

from src.src import *

# %config InlineBackend.figure_format = "retina"

In [2]:
# Read in adaptation subject data
df_adapt = pd.read_csv("modeling/adaptation-data.csv")
df_adapt.rename(columns={"fbi": "vis_fb",
                   "rotation": "delta_xv",
                   "theta_maxradv": "x_f"}, 
            inplace=True)
df_adapt["session"] = "adapt"

# Create perturbation variable
num_adapt_trials = np.max(df_adapt["TN"])
pert_trial = np.zeros(num_adapt_trials, dtype="bool")
# every odd trial beginning with Trial Num 101 (idx=100) is pert trial
pert_trial[100::2] = True
pert_trial = np.tile(pert_trial, reps=np.max(df_adapt["SN"]))
df_adapt["pert_trial"] = pert_trial


# Read in pert detection subject data
df_sdt = pd.read_csv("modeling/sdt-data.csv")
df_sdt.rename(columns={"fbi": "vis_fb",
                       "rotation": "delta_xv",
                       "theta_maxradv": "x_f",
                       "perturbation_responses": "pert_resp"}, 
            inplace=True)
df_sdt["session"] = "sdt"
# Create perturbation variable (defined differently than adapt; every non-zero rotation here)
pert_trial = np.where(np.abs(df_sdt["delta_xv"]) > 0, True, False)
df_sdt["pert_trial"] = pert_trial

In [3]:
# Outlier removal
df_adapt = test_remove_outliers(3.5,
                                df_adapt,
                                "x_f",
                                "SN",
                                "adapt",
                                abs_cutoff=22.5,
                                switch_outlier_with_mean=False
)

df_sdt = test_remove_outliers(3.5,
                              df_sdt,
                              "x_f",
                              "SN",
                              "sdt",
                              abs_cutoff=22.5,
                              switch_outlier_with_mean=False
)


# Compute adaptation metric (must be done after replacing outliers with NaNs)
pert_indices = np.flatnonzero(df_adapt["pert_trial"])
adaptation = np.zeros(len(df_adapt))
adaptation[pert_indices] = df_adapt.loc[pert_indices + 1, "x_f_clean"].to_numpy() - df_adapt.loc[pert_indices - 1, "x_f_clean"].to_numpy()
df_adapt["adaptation"] = adaptation

### Compute baseline sigma_motor using x_f_clean (potentially has nans)

In [4]:
# Create boolean mask for trials 51-100
mask_adapt = (df_adapt["TN"] >= 51) & (df_adapt["TN"] <= 100)
mask_sdt = (df_sdt["TN"] >= 51) & (df_sdt["TN"] <= 100)

# Compute baseline motor var for each subject
sigma_motors_adapt = df_adapt[mask_adapt].groupby("SN")["x_f_clean"].std()
sigma_motors_sdt = df_sdt[mask_sdt].groupby("SN")["x_f_clean"].std()

# Map those values back to every row of original SN col
df_adapt["sigma_motor_adapt"] = df_adapt["SN"].map(sigma_motors_adapt)
df_sdt["sigma_motor_sdt"] = df_sdt["SN"].map(sigma_motors_sdt)

In [5]:
# Write out to csv
df_adapt.to_csv("behavioural/data/processed-adapt.csv", index=False)

In [6]:
# Concatenated data frame
df_comb = pd.concat([df_adapt, df_sdt])

# # Subset dataframe beginning with last baseline trial for fitting
# idx = df_comb["TN"] >= 100  
# df_fit = df_comb[idx]

# Subset dataframe beginning with first pert trial for fitting 
### First pert trial number for sdt task is 107 (6 practice trials) ###
idx = ((df_comb["session"] == "adapt") & (df_comb["TN"] >= 101)) | ((df_comb["session"] == "sdt") & (df_comb["TN"] >= 107))  
df_fit = df_comb[idx].reset_index(drop=True)

In [7]:
df_fit.head(15)

,si,SN,id,tester,TN,move_cycle,hX,hY,rot_hX,rot_hY,...,pert_trial,x_f_z,x_f_outlier,no_fit,x_f_mean,x_f_clean,adaptation,sigma_motor_adapt,pert_resp,sigma_motor_sdt
0,1,1,01_i,s,101,101,-6.523699,95.708669,0.0,0.0,...,True,-1.121559,False,False,1.541759,-1.606234,0.983717,2.246088,NaN,NaN
1,1,1,01_i,s,102,102,-0.073699,96.518669,0.0,0.0,...,False,0.036549,False,False,1.541759,1.624840,0.000000,2.246088,NaN,NaN
2,1,1,01_i,s,103,103,-3.833699,93.848669,0.0,0.0,...,True,-0.591950,False,False,1.541759,-0.128646,0.581988,2.246088,NaN,NaN
3,1,1,01_i,s,104,104,0.466301,92.658669,0.0,0.0,...,False,0.245150,False,False,1.541759,2.206829,0.000000,2.246088,NaN,NaN
4,1,1,01_i,s,105,105,-0.043699,90.888669,0.0,0.0,...,True,-0.011512,False,False,1.541759,1.490751,0.870253,2.246088,NaN,NaN
5,1,1,01_i,s,106,106,2.096301,91.888669,0.0,0.0,...,False,0.557073,False,False,1.541759,3.077081,0.000000,2.246088,NaN,NaN
6,1,1,01_i,s,107,107,0.256301,94.218669,0.0,0.0,...,True,-0.305145,False,False,1.541759,0.671527,-0.190481,2.246088,NaN,NaN
7,1,1,01_i,s,108,108,0.906301,95.168669,0.0,0.0,...,False,0.488799,False,False,1.541759,2.886601,0.000000,2.246088,NaN,NaN
8,1,1,01_i,s,109,109,2.026301,93.148669,0.0,0.0,...,True,1.663108,False,False,1.541759,6.162874,-5.025827,2.246088,NaN,NaN
9,1,1,01_i,s,110,110,-4.003699,93.218669,0.0,0.0,...,False,-1.312598,False,False,1.541759,-2.139226,0.000000,2.246088,NaN,NaN


In [8]:
df_fit.to_csv("modeling/combined-data.csv", index=False)

---

In [9]:
adapt = df_fit[df_fit["session"]=="adapt"].reset_index(drop=True)
sdt = df_fit[df_fit["session"]=="sdt"].reset_index(drop=True)
adapt.head()

,si,SN,id,tester,TN,move_cycle,hX,hY,rot_hX,rot_hY,...,pert_trial,x_f_z,x_f_outlier,no_fit,x_f_mean,x_f_clean,adaptation,sigma_motor_adapt,pert_resp,sigma_motor_sdt
0,1,1,01_i,s,101,101,-6.523699,95.708669,0.0,0.0,...,True,-1.121559,False,False,1.541759,-1.606234,0.983717,2.246088,NaN,NaN
1,1,1,01_i,s,102,102,-0.073699,96.518669,0.0,0.0,...,False,0.036549,False,False,1.541759,1.624840,0.000000,2.246088,NaN,NaN
2,1,1,01_i,s,103,103,-3.833699,93.848669,0.0,0.0,...,True,-0.591950,False,False,1.541759,-0.128646,0.581988,2.246088,NaN,NaN
3,1,1,01_i,s,104,104,0.466301,92.658669,0.0,0.0,...,False,0.245150,False,False,1.541759,2.206829,0.000000,2.246088,NaN,NaN
4,1,1,01_i,s,105,105,-0.043699,90.888669,0.0,0.0,...,True,-0.011512,False,False,1.541759,1.490751,0.870253,2.246088,NaN,NaN


In [10]:
print(adapt["x_f_outlier"].sum())
print(adapt["no_fit"].sum())

205
327


In [11]:
print(adapt.groupby("SN")["x_f_outlier"].sum())
print(adapt.groupby("SN")["no_fit"].sum())

SN
1       5
2       4
3     114
4      10
5       5
6       5
7       5
8       2
9      10
10      4
11      6
12      9
13      3
14      8
15     11
16      4
Name: x_f_outlier, dtype: int64
SN
1      10
2       8
3     146
4      19
5      10
6      10
7      10
8       4
9      20
10      8
11     12
12     18
13      6
14     16
15     22
16      8
Name: no_fit, dtype: int64


In [12]:
print(adapt.groupby("SN")["sigma_motor_adapt"].mean().mean())
print(adapt.groupby("SN")["x_f"].std().describe())


2.049132738356228
count    16.000000
mean      3.630514
std       2.752563
min       1.982195
25%       2.452385
50%       2.723689
75%       3.122005
max      11.932930
Name: x_f, dtype: float64


In [13]:
print(adapt["x_f_clean"].max())

16.8144499543328


In [14]:
print(sdt.groupby("SN")["x_f_outlier"].sum())
print(sdt.groupby("SN")["no_fit"].sum())

SN
1      2
2      2
3     23
4      1
5      2
6      2
7      3
8      3
9      6
10     3
11     6
12     1
13     1
14     5
15     0
16     4
Name: x_f_outlier, dtype: int64
SN
1      2
2      2
3     23
4      1
5      2
6      2
7      3
8      3
9      6
10     3
11     6
12     1
13     1
14     5
15     0
16     4
Name: no_fit, dtype: int64
